# Report modeling figures

Generates the model-selection figures used in the report from nested cross-validation CSV outputs.

Outputs:
- Figure 4: outer-fold macro F1 comparison
- Figure 5: HistGradientBoosting hyperparameter optimization

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from util.paths import NESTED_DIR

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pipeline.py").exists() and (candidate / "models").exists():
            return candidate
    raise FileNotFoundError("Could not find project root.")


PROJECT_ROOT = find_project_root()

FIGURES_DIR = NESTED_DIR/ "nested_cv_report_figures"
GRID_DIR = NESTED_DIR / "grid_search_cv_results"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

fold_results = pd.read_csv(NESTED_DIR/ "group_nested_kfold_cv_fold_results.csv")
summary_results = pd.read_csv(NESTED_DIR/ "group_nested_kfold_cv_results.csv")
PROJECT_ROOT, FIGURES_DIR

## Figure 4

Outer-fold macro F1 distribution across candidate models.

In [ ]:
model_order = (
    summary_results.sort_values("f1_macro_pct_mean", ascending=False)["model"]
    .tolist()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), gridspec_kw={"width_ratios": [1.25, 1]})

sns.boxplot(data=fold_results, y="model", x="f1_macro_pct", order=model_order, color="#d8e2ef", ax=axes[0])
sns.stripplot(data=fold_results, y="model", x="f1_macro_pct", order=model_order, color="#1f2937", size=3, ax=axes[0])
axes[0].set_title("All models")
axes[0].set_xlabel("Macro F1-score (%)")
axes[0].set_ylabel("")

top_models = model_order[:3]
top_df = fold_results[fold_results["model"].isin(top_models)]
sns.boxplot(data=top_df, y="model", x="f1_macro_pct", order=top_models, color="#d8e2ef", ax=axes[1])
sns.stripplot(data=top_df, y="model", x="f1_macro_pct", order=top_models, color="#1f2937", size=3, ax=axes[1])
axes[1].set_title("Zoom on top models")
axes[1].set_xlabel("Macro F1-score (%)")
axes[1].set_ylabel("")
axes[1].set_xlim(top_df["f1_macro_pct"].min() - 0.15, top_df["f1_macro_pct"].max() + 0.15)

fig.suptitle("Outer-Fold Macro F1 Scores Across Candidate Models", y=1.02)
plt.tight_layout()
output_path = FIGURES_DIR / "figure_nested_cv_outer_f1_boxplot.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print(f"Saved to: {output_path}")

## Figure 5

Top HistGradientBoosting inner-CV configurations aggregated across outer folds.

In [ ]:
hgb_files = sorted(GRID_DIR.glob("histgradientboosting_outer_fold_*.csv"))
if not hgb_files:
    raise FileNotFoundError("No HistGradientBoosting grid-search CSV files found.")

grid = pd.concat([pd.read_csv(path).assign(source_file=path.name) for path in hgb_files], ignore_index=True)
param_cols = [col for col in grid.columns if col.startswith("param_model__") and not grid[col].isna().all()]

agg = (
    grid.groupby(param_cols, dropna=False)["mean_test_score"]
    .mean()
    .reset_index()
    .sort_values("mean_test_score", ascending=False)
    .head(8)
)

def clean_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, float) and value.is_integer():
        return int(value)
    return value

def format_config(row: pd.Series) -> str:
    early = clean_value(row.get("param_model__early_stopping"))
    lr = clean_value(row.get("param_model__learning_rate"))
    iters = clean_value(row.get("param_model__max_iter"))
    leaves = clean_value(row.get("param_model__max_leaf_nodes"))
    min_leaf = clean_value(row.get("param_model__min_samples_leaf"))
    l2 = clean_value(row.get("param_model__l2_regularization"))
    return f"lr={lr}, iter={iters}, leaves={leaves}, min_leaf={min_leaf}, l2={l2}, early={early}"

agg["config"] = agg.apply(format_config, axis=1)
agg["mean_inner_f1_pct"] = agg["mean_test_score"] * 100
plot_df = agg.sort_values("mean_inner_f1_pct", ascending=True)

fig, ax = plt.subplots(figsize=(10, 4.8))
sns.barplot(data=plot_df, x="mean_inner_f1_pct", y="config", color="#4e79a7", ax=ax)
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=4, fontsize=8)

ax.set_title("HistGradientBoosting Hyperparameter Optimization")
ax.set_xlabel("Mean inner-CV macro F1 (%)")
ax.set_ylabel("")
ax.set_xlim(plot_df["mean_inner_f1_pct"].min() - 0.1, plot_df["mean_inner_f1_pct"].max() + 0.15)
ax.tick_params(axis="y", labelsize=8)
sns.despine(left=True)
plt.tight_layout()
output_path = FIGURES_DIR / "figure_5_histgradientboosting_grid_search_configs.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print(f"Saved to: {output_path}")